# Neural CFR+ and neural FSP focused screen

This notebook screens the remaining high-value parameters on the 18-claim `r4_s4_h2_hp2pt_ss` calibration game.

- Neural CFR+ varies regret fitting effort and traversal volume around the reference defaults.
- Neural FSP varies outer-update frequency, best-response strength, and average-policy supervision.
- Every configuration receives the same measured GPU-training budget. Exact evaluation time is excluded.
- Neural FSP is evaluated only at completed outer-iteration boundaries.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.algo.br_exact_dense_to_dense import best_response_dense
from liars_poker.algo.deep_cfr_plus import DeepCFRPlusTrainer
from liars_poker.algo.neural_fsp import NeuralFSPTrainer
from liars_poker.core import GameSpec
from liars_poker.env import rules_for_spec
from liars_poker.eval.match_dense import evaluate_dense_response
from liars_poker.policies.action_conditioned import (
    compile_action_conditioned_q_to_dense,
    compile_action_conditioned_to_dense,
)
from liars_poker.policies.neural import compile_neural_to_dense
from liars_poker.policies.tabular_dense import mix_dense
from liars_poker.serialization import save_policy

assert torch.cuda.is_available(), 'This screen is intended for a CUDA machine.'
device = torch.device('cuda')
torch.set_float32_matmul_precision('high')
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=2,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips'),
    suit_symmetry=True,
)

SEED = 7
MINUTES_PER_CONFIGURATION = 8
CFR_PLUS_EVALUATE_EVERY_MINUTES = 2
RUN_CFR_PLUS_SCREEN = True
RUN_NEURAL_FSP_SCREEN = True
SAVE_FINAL_POLICIES = True

screen_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
OUTPUT_DIR = REPO_ROOT / 'artifacts' / 'neural_method_screens' / f'{spec.to_short_str()}___{screen_id}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('claims:', len(rules_for_spec(spec).claims))
print('output:', OUTPUT_DIR)
print('planned measured GPU-training minutes:', 8 * MINUTES_PER_CONFIGURATION)

In [ ]:
def exact_exploitability(dense_policy):
    _, meta = best_response_dense(
        spec,
        dense_policy,
        debug=False,
        store_state_values=False,
    )
    p_first, p_second = meta['computer'].exploitability()
    return float(p_first + p_second - 1.0)


def normalized_auc(frame, value_col):
    ordered = frame.sort_values('measured training s')
    x = ordered['measured training s'].to_numpy(dtype=float)
    y = ordered[value_col].to_numpy(dtype=float)
    if len(x) < 2 or x[-1] <= x[0]:
        return np.nan
    return float(np.trapezoid(y, x) / (x[-1] - x[0]))


def mean_metric(metrics, section, name):
    values = [row[name] for row in metrics[section] if row.get('records', 0)]
    return float(np.mean(values)) if values else np.nan

## 1. Neural CFR+ screen

The architecture, replay, batch size and learning rate remain fixed. The screen asks whether the reference `1024 traversals / 100 regret steps` allocation is already near the useful wall-clock frontier.

In [ ]:
CFR_PLUS_CONFIGS = {
    'reference: 1024 traversals; 100 steps': {
        'traversals_per_player': 1024,
        'regret_train_steps': 100,
    },
    'less fitting: 1024 traversals; 50 steps': {
        'traversals_per_player': 1024,
        'regret_train_steps': 50,
    },
    'more fitting: 1024 traversals; 300 steps': {
        'traversals_per_player': 1024,
        'regret_train_steps': 300,
    },
    'more data: 2048 traversals; 100 steps': {
        'traversals_per_player': 2048,
        'regret_train_steps': 100,
    },
}

CFR_PLUS_BASE = {
    'regret_hidden_sizes': (2048, 2048),
    'strategy_hidden_sizes': (512, 512),
    'device': device,
    'seed': SEED,
    'regret_buffer_capacity': 500_000,
    'strategy_buffer_capacity': 2_000_000,
    'learning_rate': 1e-3,
    'batch_size': 4096,
    'strategy_train_steps': 50,
    'regret_positive_weight': 0.5,
    'strategy_weighting': 'linear',
    'validation_fraction': 0.02,
    'validation_buffer_capacity': 20_000,
    'traversal_backend': 'gpu_native',
    'traversal_batch_size': 1024,
    'device_replay': True,
    'fused_optimizer': True,
    'amp_dtype': None,
    'compile_models': False,
}

In [ ]:
cfr_plus_frames = []
cfr_plus_training = {}

if RUN_CFR_PLUS_SCREEN:
    for name, variant in CFR_PLUS_CONFIGS.items():
        print(f'\n=== neural CFR+: {name} ===')
        trainer = DeepCFRPlusTrainer(
            spec,
            **CFR_PLUS_BASE,
            regret_train_steps=variant['regret_train_steps'],
        )
        traversals = variant['traversals_per_player']
        budget_s = MINUTES_PER_CONFIGURATION * 60.0
        evaluate_every_s = CFR_PLUS_EVALUATE_EVERY_MINUTES * 60.0
        measured_s = 0.0
        next_evaluation_s = 0.0
        rows = []
        training_records = []

        while measured_s < budget_s:
            if measured_s >= next_evaluation_s:
                average_dense = compile_neural_to_dense(
                    trainer.average_policy(), batch_size=65_536,
                )
                current_dense = trainer.current_policy_dense(batch_size=65_536)
                validation = trainer.validation_metrics()
                row = {
                    'method': 'neural CFR+',
                    'configuration': name,
                    'measured training s': measured_s,
                    'iteration': trainer.iteration,
                    'learned average exploitability': exact_exploitability(average_dense),
                    'current exploitability': exact_exploitability(current_dense),
                    'regret validation MSE': mean_metric(validation, 'regret', 'mse'),
                    'regret-induced strategy TV': mean_metric(validation, 'regret', 'strategy_tv'),
                    'average-network strategy TV': mean_metric(validation, 'strategy', 'strategy_tv'),
                }
                rows.append(row)
                print(
                    f"train={measured_s / 60:5.1f}m iter={trainer.iteration:4d} "
                    f"average={row['learned average exploitability']:.6f} "
                    f"current={row['current exploitability']:.6f}"
                )
                while next_evaluation_s <= measured_s:
                    next_evaluation_s += evaluate_every_s
                del average_dense, current_dense
                gc.collect()

            start = time.perf_counter()
            record = trainer.run_iteration(traversals_per_player=traversals)
            measured_s += time.perf_counter() - start
            training_records.append(record)

        average_dense = compile_neural_to_dense(
            trainer.average_policy(), batch_size=65_536,
        )
        current_dense = trainer.current_policy_dense(batch_size=65_536)
        validation = trainer.validation_metrics()
        rows.append({
            'method': 'neural CFR+',
            'configuration': name,
            'measured training s': measured_s,
            'iteration': trainer.iteration,
            'learned average exploitability': exact_exploitability(average_dense),
            'current exploitability': exact_exploitability(current_dense),
            'regret validation MSE': mean_metric(validation, 'regret', 'mse'),
            'regret-induced strategy TV': mean_metric(validation, 'regret', 'strategy_tv'),
            'average-network strategy TV': mean_metric(validation, 'strategy', 'strategy_tv'),
        })
        frame = pd.DataFrame(rows).drop_duplicates(
            subset=['configuration', 'iteration'], keep='last',
        )
        cfr_plus_frames.append(frame)
        cfr_plus_training[name] = training_records
        frame.to_csv(OUTPUT_DIR / f"cfr_plus_{len(cfr_plus_frames):02d}.csv", index=False)
        if SAVE_FINAL_POLICIES:
            save_policy(
                trainer.average_policy(),
                str(OUTPUT_DIR / 'policies' / f'cfr_plus_{len(cfr_plus_frames):02d}'),
            )
        del trainer, average_dense, current_dense
        gc.collect()
        torch.cuda.empty_cache()

cfr_plus_results = (
    pd.concat(cfr_plus_frames, ignore_index=True)
    if cfr_plus_frames else pd.DataFrame()
)
cfr_plus_results

In [ ]:
if not cfr_plus_results.empty:
    cfr_plus_summary_rows = []
    for name, frame in cfr_plus_results.groupby('configuration', sort=False):
        final = frame.sort_values('measured training s').iloc[-1]
        records = cfr_plus_training[name]
        cfr_plus_summary_rows.append({
            'configuration': name,
            'iterations completed': int(final['iteration']),
            'final average exploitability': final['learned average exploitability'],
            'best average exploitability': frame['learned average exploitability'].min(),
            'average normalized AUC': normalized_auc(frame, 'learned average exploitability'),
            'final current exploitability': final['current exploitability'],
            'final regret MSE': final['regret validation MSE'],
            'final regret strategy TV': final['regret-induced strategy TV'],
            'mean traversal s': np.mean([row['timing']['traversal_s'] for row in records]),
            'mean regret fit s': np.mean([row['timing']['regret_training_s'] for row in records]),
            'mean strategy fit s': np.mean([row['timing']['strategy_training_s'] for row in records]),
        })
    cfr_plus_summary = pd.DataFrame(cfr_plus_summary_rows).set_index('configuration')
    display(cfr_plus_summary.style.format(precision=6))

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
    for name, frame in cfr_plus_results.groupby('configuration', sort=False):
        x = frame['measured training s'] / 60.0
        axes[0].plot(x, frame['learned average exploitability'], marker='o', label=name)
        axes[1].plot(x, frame['current exploitability'], marker='o', label=name)
        axes[2].plot(x, frame['regret validation MSE'], marker='o', label=name)
    for ax, title, ylabel in [
        (axes[0], 'Neural CFR+ learned average', 'Exploitability'),
        (axes[1], 'Neural CFR+ current strategy', 'Exploitability'),
        (axes[2], 'Current-update regret fit', 'MSE'),
    ]:
        ax.set(title=title, xlabel='Measured GPU-training minutes', ylabel=ylabel)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    axes[0].set_yscale('log')
    axes[1].set_yscale('log')
    fig.tight_layout();

## 2. Neural FSP outer-loop screen

The neural architectures and inner fitted-return settings remain fixed. The four configurations isolate:

1. frequent, weaker best-response updates;
2. the current balanced default;
3. fewer but substantially stronger best responses;
4. extra sampling and fitting for the average-policy network.

For every completed phase, the notebook records the exact exploitability of the learned average, the exact value found by the neural BR against the frozen pre-update average, and the exact reach-correct mixture of generated BR policies.

In [ ]:
FSP_CONFIGS = {
    'frequent updates: 50 BR iterations': {
        'br_iterations': 50,
        'br_episodes_per_role': 512,
        'strategy_episodes_per_role': 512,
        'strategy_train_steps': 100,
    },
    'balanced: 100 BR iterations': {
        'br_iterations': 100,
        'br_episodes_per_role': 512,
        'strategy_episodes_per_role': 512,
        'strategy_train_steps': 100,
    },
    'strong BR: 300 BR iterations': {
        'br_iterations': 300,
        'br_episodes_per_role': 512,
        'strategy_episodes_per_role': 512,
        'strategy_train_steps': 100,
    },
    'average-heavy: 100 BR iterations': {
        'br_iterations': 100,
        'br_episodes_per_role': 512,
        'strategy_episodes_per_role': 2048,
        'strategy_train_steps': 300,
    },
}

FSP_TRAINER_KWARGS = {
    'average_state_hidden_sizes': (512, 512),
    'average_action_hidden_sizes': (128, 128),
    'average_embedding_dim': 256,
    'strategy_buffer_capacity': 1_000_000,
    'validation_buffer_capacity': 20_000,
    'validation_fraction': 0.02,
    'strategy_batch_size': 4096,
    'strategy_train_steps': 100,
    'strategy_learning_rate': 1e-3,
    'br_kwargs': {
        'state_hidden_sizes': (512, 512),
        'action_hidden_sizes': (128, 128),
        'embedding_dim': 256,
        'replay_capacity': 1_000_000,
        'batch_size': 4096,
        'learning_rate': 1e-3,
        'train_steps': 100,
        'warmup_transitions': 20_000,
        'epsilon_start': 0.25,
        'epsilon_end': 0.05,
        'epsilon_decay_decisions': 250_000,
        'rollouts_per_action': 1,
        'fused_optimizer': True,
    },
    'device': device,
    'fused_optimizer': True,
    'seed': SEED,
}
FSP_FIXED_RUN_KWARGS = {
    'br_rollout_batch_size': 256,
    'strategy_collection_batch_size': 128,
}

In [ ]:
fsp_frames = []

if RUN_NEURAL_FSP_SCREEN:
    for name, variant in FSP_CONFIGS.items():
        print(f'\n=== neural FSP: {name} ===')
        trainer = NeuralFSPTrainer(spec, **FSP_TRAINER_KWARGS)
        budget_s = MINUTES_PER_CONFIGURATION * 60.0
        measured_s = 0.0
        rows = []

        learned_dense = compile_action_conditioned_to_dense(
            trainer.average_policy(), batch_size=65_536,
        )
        generated_dense = learned_dense
        learned_exploitability = exact_exploitability(learned_dense)
        rows.append({
            'method': 'neural FSP',
            'configuration': name,
            'measured training s': 0.0,
            'outer iteration': 0,
            'learned average exploitability': learned_exploitability,
            'generated average exploitability': learned_exploitability,
            'learned minus generated': 0.0,
            'neural BR discovered exploitability': np.nan,
            'exact BR ceiling': learned_exploitability,
            'neural BR oracle gap': np.nan,
            'BR training s': 0.0,
            'average collection s': 0.0,
            'average fitting s': 0.0,
            'average validation cross entropy': np.nan,
            'average validation action accuracy': np.nan,
        })

        while measured_s < budget_s:
            frozen_dense = learned_dense
            exact_ceiling = learned_exploitability

            start = time.perf_counter()
            record = trainer.run_iteration(
                br_iterations=variant['br_iterations'],
                br_episodes_per_role=variant['br_episodes_per_role'],
                strategy_episodes_per_role=variant['strategy_episodes_per_role'],
                strategy_train_steps=variant['strategy_train_steps'],
                **FSP_FIXED_RUN_KWARGS,
            )
            measured_s += time.perf_counter() - start

            br_dense = compile_action_conditioned_q_to_dense(
                trainer.best_response_policy(), batch_size=65_536,
            )
            found_first, found_second = evaluate_dense_response(
                spec, frozen_dense, br_dense,
            )
            found_br = float(found_first + found_second - 1.0)
            eta = 1.0 / (trainer.iteration + 1.0)
            generated_dense = mix_dense(br_dense, generated_dense, eta)

            learned_dense = compile_action_conditioned_to_dense(
                trainer.average_policy(), batch_size=65_536,
            )
            learned_exploitability = exact_exploitability(learned_dense)
            validation = record['validation']
            valid = [row for row in validation if row.get('records', 0)]
            row = {
                'method': 'neural FSP',
                'configuration': name,
                'measured training s': measured_s,
                'outer iteration': trainer.iteration,
                'learned average exploitability': learned_exploitability,
                'generated average exploitability': np.nan,
                'learned minus generated': np.nan,
                'neural BR discovered exploitability': found_br,
                'exact BR ceiling': exact_ceiling,
                'neural BR oracle gap': exact_ceiling - found_br,
                'BR training s': record['br_s'],
                'average collection s': record['strategy_collect_s'],
                'average fitting s': record['strategy_fit_s'],
                'average validation cross entropy': float(np.mean([
                    item['cross_entropy'] for item in valid
                ])) if valid else np.nan,
                'average validation action accuracy': float(np.mean([
                    item['sampled_action_accuracy'] for item in valid
                ])) if valid else np.nan,
            }
            rows.append(row)
            print(
                f"outer={trainer.iteration:2d} train={measured_s / 60:5.1f}m "
                f"average={learned_exploitability:.6f} "
                f"BR gap={exact_ceiling - found_br:.6f}"
            )

        generated_exploitability = exact_exploitability(generated_dense)
        rows[-1]['generated average exploitability'] = generated_exploitability
        rows[-1]['learned minus generated'] = (
            rows[-1]['learned average exploitability'] - generated_exploitability
        )
        frame = pd.DataFrame(rows)
        fsp_frames.append(frame)
        frame.to_csv(OUTPUT_DIR / f"neural_fsp_{len(fsp_frames):02d}.csv", index=False)
        if SAVE_FINAL_POLICIES:
            save_policy(
                trainer.average_policy(),
                str(OUTPUT_DIR / 'policies' / f'neural_fsp_{len(fsp_frames):02d}'),
            )
        del trainer, learned_dense, generated_dense, frozen_dense, br_dense
        gc.collect()
        torch.cuda.empty_cache()

fsp_results = (
    pd.concat(fsp_frames, ignore_index=True)
    if fsp_frames else pd.DataFrame()
)
fsp_results

In [ ]:
if not fsp_results.empty:
    fsp_summary_rows = []
    for name, frame in fsp_results.groupby('configuration', sort=False):
        final = frame.sort_values('measured training s').iloc[-1]
        noninitial = frame[frame['outer iteration'] > 0]
        fsp_summary_rows.append({
            'configuration': name,
            'outer iterations completed': int(final['outer iteration']),
            'actual measured training min': final['measured training s'] / 60.0,
            'final learned-average exploitability': final['learned average exploitability'],
            'best learned-average exploitability': frame['learned average exploitability'].min(),
            'learned-average normalized AUC': normalized_auc(frame, 'learned average exploitability'),
            'final generated-average exploitability': final['generated average exploitability'],
            'final learned-minus-generated': final['learned minus generated'],
            'final neural BR oracle gap': final['neural BR oracle gap'],
            'mean neural BR oracle gap': noninitial['neural BR oracle gap'].mean(),
            'mean BR training s': noninitial['BR training s'].mean(),
            'mean average collection s': noninitial['average collection s'].mean(),
            'mean average fitting s': noninitial['average fitting s'].mean(),
        })
    fsp_summary = pd.DataFrame(fsp_summary_rows).set_index('configuration')
    display(fsp_summary.style.format(precision=6))

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
    for name, frame in fsp_results.groupby('configuration', sort=False):
        x = frame['measured training s'] / 60.0
        axes[0].plot(x, frame['learned average exploitability'], marker='o', label=name)
        active = frame['outer iteration'] > 0
        axes[1].plot(x[active], frame.loc[active, 'neural BR oracle gap'], marker='o', label=name)
        axes[2].plot(x[active], frame.loc[active, 'BR training s'], marker='o', label=f'{name}: BR')
        axes[2].plot(x[active], frame.loc[active, 'average fitting s'], marker='x', linestyle='--', label=f'{name}: average')
    for ax, title, ylabel in [
        (axes[0], 'Neural FSP learned average', 'Exploitability'),
        (axes[1], 'Best-response approximation error', 'Exact oracle gap'),
        (axes[2], 'Outer-loop compute allocation', 'Seconds'),
    ]:
        ax.set(title=title, xlabel='Measured GPU-training minutes', ylabel=ylabel)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=7)
    axes[0].set_yscale('log')
    axes[1].set_yscale('log')
    fig.tight_layout();

## Interpretation guide

For neural CFR+, prefer low learned-average AUC and final exploitability. Current-strategy quality and regret MSE are diagnostic rather than selection criteria.

For neural FSP:

- a large BR oracle gap means the outer update is based on a weak response;
- a large learned-minus-generated gap means average-policy distillation is the bottleneck;
- a low generated-average exploitability but poor learned average calls for more average-policy data/fitting;
- a strong BR configuration that completes too few outer updates may lose per unit wall time despite producing better individual responses.